# Reference Frames And Transforms

This notebook explains the first semantic layer of PyTex: named frames, explicit
domains, and reusable rigid transforms. The goal is not only to show the API, but
to fix the meaning of every vector before later orientation, EBSD, or diffraction
workflows are attempted.

## Learning Goals

- construct `ReferenceFrame` objects and understand frame domains
- express rigid mappings with `FrameTransform`
- work with `VectorSet` so frame meaning stays attached to batched data
- connect the code to the mathematical map

## Mathematical Contract

PyTex treats a frame transform as

$$
\mathbf{v}^{(t)} = \mathbf{R}_{s\rightarrow t}\,\mathbf{v}^{(s)} + \mathbf{t}_{s\rightarrow t},
$$

where the source and target frames are explicit objects, not comments or naming
conventions.


In [ ]:
import numpy as np

from pytex import (
    AcquisitionGeometry,
    BenchmarkManifest,
    CalibrationRecord,
    CrystalMap,
    CrystalPlane,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    ReferenceFrame,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
)


def make_context():
    crystal = ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    symmetry = SymmetrySpec.from_point_group("m-3m", reference_frame=crystal)
    lattice = Lattice(3.6, 3.6, 3.6, 90.0, 90.0, 90.0, crystal_frame=crystal)
    phase = Phase("fcc_demo", lattice=lattice, symmetry=symmetry, crystal_frame=crystal)
    return crystal, specimen, map_frame, detector, lab, phase


In [ ]:
crystal, specimen, map_frame, detector, lab, phase = make_context()

specimen_to_map = FrameTransform(
    source=specimen,
    target=map_frame,
    rotation_matrix=np.eye(3),
    translation_vector=np.array([0.0, 0.0, 0.0]),
)

vectors = VectorSet(
    values=np.array(
        [
            [1.0, 0.0, 0.0],
            [0.0, 1.0, 0.0],
            [0.0, 0.0, 1.0],
        ]
    ),
    reference_frame=specimen,
)

mapped = specimen_to_map.apply_to_vectors(vectors)

print(vectors.reference_frame.name, "->", mapped.reference_frame.name)
print(mapped.values)


## Why The `VectorSet` Matters

A raw `(n, 3)` array is not enough when the frame matters. PyTex therefore treats
batched vectors as a first-class semantic object:

- the storage remains NumPy-backed and vectorized
- the shared frame is explicit
- later transforms can reject mismatches early

This is the pattern that the rest of the library follows: preserve vectorization,
but never let the science collapse back into anonymous arrays.


In [ ]:
try:
    wrong_vectors = VectorSet(values=np.array([[1.0, 0.0, 0.0]]), reference_frame=crystal)
    specimen_to_map.apply_to_vectors(wrong_vectors)
except ValueError as exc:
    print(type(exc).__name__)
    print(exc)
